In [3]:
# Core Python libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import random

# TensorFlow / Keras essentials
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, callbacks, optimizers
from tensorflow.keras.preprocessing import image_dataset_from_directory

# For class weight calculation
from sklearn.utils import class_weight

# For mounting Google Drive
from google.colab import drive

# Check version (optional but good practice)
print("TensorFlow version:", tf.__version__)
print("GPU Available:", "Yes" if tf.config.list_physical_devices('GPU') else "No")

TensorFlow version: 2.20.0
GPU Available: No


In [4]:
# Mount your Google Drive to access the dataset
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# List everything inside your main Google Drive folder
base_path = '/content/drive/MyDrive/'
print("Files and folders in your MyDrive:")
for item in os.listdir(base_path):
    print(f" - {item}")

# Set the correct path to your dataset.
# This is already correct for your folder!
dataset_path = base_path + 'Dog_Skin_Disease_Final_Dataset'

# Verify the path exists and check the subfolders (which are your classes)
if os.path.exists(dataset_path):
    print(f"\n✅ Dataset found at: {dataset_path}")
    print("Subfolders (your image classes):", os.listdir(dataset_path))
else:
    print(f"\n❌ Folder not found at {dataset_path}. Please copy the correct path from the list above.")

Files and folders in your MyDrive:
 - Classroom
 - C S.gdoc
 - Share Market.gdoc
 - New words Dictionary .gdoc
 - Video recording 
 - 4.gdoc
 - 90's final.gvid
 - Untitled document (15).gdoc
 - Gaurab Pokhrel's Cover Letter.gdoc
 - Roadmap for Data visualization.gdoc
 - Report gallery.gdoc
 - Today.m4a
 - Untitled document (14).gdoc
 - Computer netowrk.gdoc
 - IPsec vs SSL comparision.gdoc
 - Booklet of 15 point CNS.gdoc
 - Test and exam same question ?.gdoc
 - TCP IP Model.gdoc
 - Comparison: OSI vs. TCP IP.gdoc
 - Circuit vs Packet switching.gdoc
 - IPv4 vs IPv6.gdoc
 - Checksum vs cyclic redundency check.gdoc
 - Data visualization.pdf
 - Data visualization.txt
 - Data visualization.gdoc
 - CID 10 questions..gdoc
 - CID EXAM.gdoc
 - Untitled document (13).gdoc
 - Group Assignment work division.gdoc
 - Email For esewa Account Delete.gdoc
 - DV.gsheet
 - ITS65504_Group Assignment_MLO2_202601_with_rubrics.gdoc
 - Task 4 work.gdoc
 - Untitled document (12).gdoc
 - ITS68404_0381121_INDVAS

In [10]:
# Data augmentation pipeline
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# This will be inserted at the beginning of the EfficientNet model later.

In [12]:
# Load the dataset from train/ and valid/ folders
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, callbacks, optimizers

train_dir = '/content/drive/MyDrive/Dog_Skin_Disease_Final_Dataset/train'
valid_dir = '/content/drive/MyDrive/Dog_Skin_Disease_Final_Dataset/valid'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=123
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    valid_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)   # <-- This is what you need

print(f"Classes: {class_names}")
print(f"Number of classes: {num_classes}")

Found 2974 files belonging to 6 classes.
Found 839 files belonging to 6 classes.
Classes: ['Dermatitis', 'Fungal_infections', 'Healthy', 'Hypersensitivity', 'demodicosis', 'ringworm']
Number of classes: 6


In [13]:
def build_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')  # num_classes is now defined
    ])
    return model

cnn_model = build_cnn()
cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,734 (42.61 MB)

 Trainable params: 11,169,734 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# ---------- Data Augmentation (already defined) ----------
# If you haven't defined it yet:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# ---------- Build EfficientNet-B0 model ----------
def build_efficientnet():
    # Load pre-trained base (without top classification layer)
    base = applications.EfficientNetB0(weights='imagenet',
                                       include_top=False,
                                       input_shape=(224, 224, 3))
    # Stage 1: freeze base
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    # Apply data augmentation (only during training, Keras handles this)
    x = data_augmentation(inputs)
    # Pass through the frozen base
    x = base(x, training=False)   # training=False ensures BatchNorm layers stay in inference mode
    # Custom classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

# Build and compile
eff_model = build_efficientnet()
eff_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

eff_model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_2 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,057,257 (15.48 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [15]:
from sklearn.utils import class_weight
import numpy as np

# Extract all training labels
y_train = np.concatenate([y for x, y in train_ds], axis=0)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(weights))
print("Class Weights:", class_weight_dict)

Class Weights: {0: np.float64(0.909480122324159), 1: np.float64(1.420248328557784), 2: np.float64(1.0074525745257452), 3: np.float64(2.3716108452950557), 4: np.float64(0.8429705215419501), 5: np.float64(0.6266329540665824)}


In [16]:
callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-7
    ),
    callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [17]:
# ---------- Hyperparameter Definitions ----------
IMG_SIZE = (224, 224)      # Input resolution
BATCH_SIZE = 32            # Batch size
LEARNING_RATE_STAGE1 = 1e-3   # Default for Adam
LEARNING_RATE_STAGE2 = 1e-5   # Low rate for fine-tuning
NUM_EPOCHS_STAGE1 = 50
NUM_EPOCHS_STAGE2 = 30

# Data loading applies IMG_SIZE and BATCH_SIZE
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=123
)

# Model compilation applies optimiser and loss
eff_model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE_STAGE1),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

Found 2974 files belonging to 6 classes.


In [22]:
# ---------- Compute Class Weights ----------
from sklearn.utils import class_weight
import numpy as np

# Get all labels from the training set
y_train = np.concatenate([y for x, y in train_ds], axis=0)
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(weights))
print("Class Weights:", class_weight_dict)

# ---------- Define Callbacks ----------
my_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

# ---------- SAFETY: set epochs to smaller values for testing ----------
# Use these for a quick test; restore to 50 and 30 for the final run.
EPOCHS_PHASE1 = 5      # originally 50
EPOCHS_PHASE2 = 5      # originally 30
EPOCHS_CNN     = 5      # originally 50

# ---------- Phase 1: Feature Extraction ----------
print(">>> PHASE 1: Training with frozen base (quick test)")
history_phase1 = eff_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=my_callbacks,
    class_weight=class_weight_dict
)

# ---------- Phase 2: Fine-Tuning ----------
print(">>> PHASE 2: Fine-tuning with unfrozen base (quick test)")

# Identify the EfficientNetB0 layer by name
base_layer = None
for layer in eff_model.layers:
    if layer.name == 'efficientnetb0':   # default name when using applications.EfficientNetB0
        base_layer = layer
        break

if base_layer is None:
    raise ValueError("EfficientNetB0 layer not found. Check layer names: " + str([layer.name for layer in eff_model.layers]))

# Unfreeze the base
base_layer.trainable = True

# Recompile with a lower learning rate
eff_model.compile(optimizer=optimizers.Adam(learning_rate=1e-5),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# Continue training
history_phase2 = eff_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=my_callbacks,
    class_weight=class_weight_dict
)

# ---------- Baseline CNN Training ----------
print(">>> Training Baseline CNN (quick test)")
history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=my_callbacks,
    class_weight=class_weight_dict
)

# ---------- Model Selection: Load Best from Checkpoint ----------
best_model = keras.models.load_model('best_model.keras')
val_loss, val_acc = best_model.evaluate(val_ds)
print(f"Best Validation Accuracy: {val_acc:.4f}")

Class Weights: {0: np.float64(0.909480122324159), 1: np.float64(1.420248328557784), 2: np.float64(1.0074525745257452), 3: np.float64(2.3716108452950557), 4: np.float64(0.8429705215419501), 5: np.float64(0.6266329540665824)}
>>> PHASE 1: Training with frozen base (quick test)
Epoch 1/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 353s 4s/step - accuracy: 0.8655 - loss: 0.4331 - val_accuracy: 0.8689 - val_loss: 0.3958 - learning_rate: 0.0010
Epoch 2/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 350s 4s/step - accuracy: 0.8692 - loss: 0.4200 - val_accuracy: 0.8749 - val_loss: 0.3876 - learning_rate: 0.0010
Epoch 3/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 351s 4s/step - accuracy: 0.8722 - loss: 0.3867 - val_accuracy: 0.8760 - val_loss: 0.3716 - learning_rate: 0.0010
Epoch 4/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 352s 4s/step - accuracy: 0.8833 - loss: 0.3763 - val_accuracy: 0.8737 - val_loss: 0.3681 - learning_rate: 0.0010
Epoch 5/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 391s 4s/step - accuracy: 0.8803 - loss: 0.3784 - val_accuracy: 0.8760 - val_loss: 0.364

In [23]:
# ---------- 3.7 Select and Save the Best Model ----------

print("=" * 50)
print("LOADING THE BEST SAVED MODEL FROM CHECKPOINT")
print("=" * 50)

# 1. Load the best model saved by ModelCheckpoint
best_model = keras.models.load_model('best_model.keras')
print("✅ Best model loaded from 'best_model.keras'")

# 2. Final evaluation on the validation set
print("\n>>> Performing final evaluation on validation set...")
val_loss, val_acc = best_model.evaluate(val_ds, verbose=1)
print(f"\n📊 Final Validation Loss:     {val_loss:.4f}")
print(f"📊 Final Validation Accuracy: {val_acc:.4f}")

# 3. Save the final model for deployment (Gradio/Streamlit demo)
best_model.save('final_deployment_model.keras')
print("\n✅ Final deployment model saved as 'final_deployment_model.keras'")

# 4. Save class names in the correct order (for the demo)
import os
with open('class_names.txt', 'w') as f:
    for name in class_names:
        f.write(f"{name}\n")
print("✅ Class names saved to 'class_names.txt'")

# 5. (Optional) Print a performance summary for your report
print("\n" + "=" * 50)
print("📋 MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"Number of Classes: {num_classes}")
print(f"Class Names:       {class_names}")
print(f"Validation Accuracy: {val_acc:.4f} ({val_acc * 100:.2f}%)")
print(f"Validation Loss:   {val_loss:.4f}")
print("=" * 50)

# 6. (Optional) Quick test on a few validation images
print("\n>>> Testing on 5 random validation samples...")
sample_images, sample_labels = next(iter(val_ds.take(1)))
predictions = best_model.predict(sample_images[:5])
predicted_classes = np.argmax(predictions, axis=1)

for i in range(5):
    true_label = class_names[sample_labels[i].numpy()]
    pred_label = class_names[predicted_classes[i]]
    confidence = np.max(predictions[i]) * 100
    print(f"Sample {i+1}: True={true_label}, Pred={pred_label}, Confidence={confidence:.1f}%")

LOADING THE BEST SAVED MODEL FROM CHECKPOINT
✅ Best model loaded from 'best_model.keras'

>>> Performing final evaluation on validation set...
27/27 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.8760 - loss: 0.3716

📊 Final Validation Loss:     0.3716
📊 Final Validation Accuracy: 0.8760

✅ Final deployment model saved as 'final_deployment_model.keras'
✅ Class names saved to 'class_names.txt'

📋 MODEL PERFORMANCE SUMMARY
Number of Classes: 6
Class Names:       ['Dermatitis', 'Fungal_infections', 'Healthy', 'Hypersensitivity', 'demodicosis', 'ringworm']
Validation Accuracy: 0.8760 (87.60%)
Validation Loss:   0.3716

>>> Testing on 5 random validation samples...
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Sample 1: True=Dermatitis, Pred=Dermatitis, Confidence=98.5%
Sample 2: True=Dermatitis, Pred=Dermatitis, Confidence=87.3%
Sample 3: True=Dermatitis, Pred=Dermatitis, Confidence=86.2%
Sample 4: True=Dermatitis, Pred=Dermatitis, Confidence=93.2%
Sample 5: True=Dermatitis, Pred=Dermatitis, Confide